# Simulate Event Histories With `jact`

This notebook builds a small semi-Markov illness-death model and samples continuous event histories from it. Along the way, it shows how to:

- simulate several individuals and independent replicates in one call,
- inspect compact trajectories and fixed-shape result arrays,
- plot sampled state paths, and
- compare simulated final-state frequencies with `model.solve(...)`.

The example is deliberately small enough to run comfortably on a CPU.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import jact

## Define a semi-Markov model

The model has three states. A healthy person can become disabled or die; a disabled person can recover or die. The recovery intensity depends on time already spent disabled, so the model is semi-Markov rather than purely Markov.

Every intensity receives calendar time `t`, source-state duration `d`, and the covariates supplied to `simulate(...)` or `solve(...)`. Here `age` has one value per individual.

In [ ]:
state_space = jact.StateSpace(
    states=["healthy", "disabled", "dead"],
    transitions=[
        ("healthy", "disabled"),
        ("healthy", "dead"),
        ("disabled", "healthy"),
        ("disabled", "dead"),
    ],
)


def disability_onset(t, d, *, age):
    attained_age = age[:, None] + t
    return 0.025 * jnp.exp(0.035 * (attained_age - 55.0)) + jnp.zeros_like(d)


def healthy_mortality(t, d, *, age):
    attained_age = age[:, None] + t
    return 0.006 * jnp.exp(0.070 * (attained_age - 55.0)) + jnp.zeros_like(d)


def recovery(t, d, *, age):
    del t, age
    return 0.45 * jnp.exp(-0.80 * d)


def disabled_mortality(t, d, *, age):
    attained_age = age[:, None] + t
    duration_multiplier = 1.0 + 0.40 * jnp.minimum(d, 5.0)
    return 0.012 * jnp.exp(0.070 * (attained_age - 55.0)) * duration_multiplier


model = state_space.build(
    transitions={
        ("healthy", "disabled"): disability_onset,
        ("healthy", "dead"): healthy_mortality,
        ("disabled", "healthy"): recovery,
        ("disabled", "dead"): disabled_mortality,
    }
)

## Simulate a small portfolio

`replicates` requests independent trajectories for every individual. `max_jumps` reserves a fixed event buffer for each trajectory, and `key` makes the random sample reproducible. Calendar-time and duration cells both have width `1 / steps_per_unit`; intensities are constant within a cell, but the sampled event times are continuous.

The first call includes JAX compilation time. Later calls with the same array shapes are typically faster.

In [ ]:
ages = jnp.array([45.0, 60.0, 75.0])
horizon = 10
steps_per_unit = 12

simulated = model.simulate(
    initial="healthy",
    horizon=horizon,
    steps_per_unit=steps_per_unit,
    max_jumps=16,
    replicates=2_000,
    key=jax.random.key(42),
    age=ages,
)

print("Reachable states:", simulated.states)
print("jump_times shape:", simulated.jump_times.shape)
print("state_path shape:", simulated.state_path.shape)
print("jump_count shape:", simulated.jump_count.shape)
print("Any overflow:", bool(jnp.any(simulated.overflow)))

All public arrays begin with `(individual, replicate, ...)`. Unused jump slots contain `NaN`, and unused state-path slots contain `-1`. `valid_mask()` identifies populated jump slots without relying on those sentinels.

In [ ]:
valid_jumps = simulated.valid_mask()
event_times = np.asarray(simulated.jump_times)[np.asarray(valid_jumps)]

print(f"Recorded jumps: {event_times.size:,}")
print(f"Mean jumps per trajectory: {float(jnp.mean(simulated.jump_count)):.3f}")
if event_times.size:
    print(f"First event time: {event_times.min():.3f} years")
    print(f"Last event time:  {event_times.max():.3f} years")

one_path = simulated.path(individual=1, replicate=0)
one_path

`path(...)` removes unused buffer entries and translates state indices to names. For analysis across all trajectories, the fixed-shape JAX arrays are usually more convenient. If pandas is installed, `simulated.to_pandas()` also produces one long-form row per jump.

## Plot individual trajectories

The plot below shows a handful of replicates for the 60-year-old individual. Each vertical change is a sampled event. Some paths have no event during the horizon.

In [ ]:
individual = 1
fig, ax = plt.subplots(figsize=(9, 5))

for replicate in range(10):
    path = simulated.path(individual, replicate)
    times = np.concatenate(([0.0], path["jump_times"], [horizon]))
    states = np.concatenate((path["state_indices"], [path["final_state"]]))
    ax.step(times, states, where="post", alpha=0.75, label=f"replicate {replicate}")

ax.set(
    xlabel="Years since entry",
    ylabel="State",
    title=f"Sample paths for entry age {float(ages[individual]):.0f}",
    xlim=(0, horizon),
    yticks=range(len(simulated.states)),
    yticklabels=simulated.states,
)
ax.grid(axis="x", alpha=0.25)
ax.legend(ncol=2, fontsize=8, frameon=False)
plt.show()

## Validate simulation against the solver

`simulate()` and `solve()` use the same midpoint-discretized intensities. The solver returns expected state probabilities; simulation produces a Monte Carlo sample whose final-state frequencies should be close to those probabilities.

In [ ]:
solved = model.solve(
    initial="healthy",
    horizon=horizon,
    steps_per_unit=steps_per_unit,
    age=ages,
)

simulated_probability = jax.nn.one_hot(
    simulated.final_state, len(simulated.states)
).mean(axis=1)
solved_probability = solved.probability[-1]

for individual, age in enumerate(np.asarray(ages)):
    print(f"Entry age {age:.0f}")
    for state_index, state in enumerate(simulated.states):
        monte_carlo = float(simulated_probability[individual, state_index])
        expected = float(solved_probability[individual, state_index])
        print(
            f"  {state:8s} simulated={monte_carlo:.3f}  "
            f"solved={expected:.3f}  difference={monte_carlo - expected:+.3f}"
        )

In [ ]:
fig, axes = plt.subplots(1, len(ages), figsize=(12, 3.5), sharey=True)
x = np.arange(len(simulated.states))

for individual, ax in enumerate(axes):
    ax.bar(
        x - 0.18,
        np.asarray(solved_probability[individual]),
        width=0.36,
        label="solve",
    )
    ax.bar(
        x + 0.18,
        np.asarray(simulated_probability[individual]),
        width=0.36,
        label="simulate",
    )
    ax.set(
        title=f"Entry age {float(ages[individual]):.0f}",
        xticks=x,
        xticklabels=simulated.states,
        ylim=(0, 1),
    )
    ax.tick_params(axis="x", rotation=25)

axes[0].set_ylabel(f"State probability at year {horizon}")
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()

## Practical choices

- Increase `replicates` to reduce Monte Carlo noise. This expands trajectories while preserving the individual batch axis.
- Increase `steps_per_unit` when intensity variation within a time or duration cell matters. Both simulation and solving become more expensive.
- Choose `max_jumps` generously, then check `simulated.overflow`. With `overflow="raise"`, any overflow raises a `RuntimeError` after simulation.
- Reuse the same random key for an exactly reproducible sample; use a new key for an independent sample.
- Pass `devices=N` to shard individuals over `N` local JAX devices when the portfolio is large enough to benefit.